# Caso G · 03 Calidad sobre la capa oro (datasets ML)

> _Tutorial · Caso de uso: **G — Calidad con agentes** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Reglas sobre datasets de entrenamiento: balance de clases, sin leakage, distribución train/test similar.


## 2. Qué se aprende

- Detección de leakage temporal y por features.
- KL divergence train vs test.
- Balance de clases.


## 3. Contexto del caso de uso

Auditar oros de B/C/D.


## 4. Relación con CENTINELA+

Pre-deploy gating.


## 5. Relación con Medallion

Oro.


## 6. Datos de entrada

Features Caso B y D si existen.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos features B (con fallback inline).


In [ ]:
parquet = ROOT / "output" / "case_B" / "features_b0.parquet"
if parquet.exists():
    X = pd.read_parquet(parquet)
else:
    df, _ = mocks.make_bdg2_education_subset()
    df = df[df.building_id == df.building_id.unique()[0]].set_index("timestamp")
    X = pd.DataFrame({
        "y": df["power_kw"],
        "t_outdoor": df["t_outdoor"],
        "lag_24h": df["power_kw"].shift(24),
    }).dropna()
print(X.shape)


## 10. Exploración paso a paso

Split temporal y comparación.


In [ ]:
n = len(X); i = int(n * 0.8)
tr, te = X.iloc[:i], X.iloc[i:]
desc_tr = tr.describe().T[["mean", "std"]].add_suffix("_tr")
desc_te = te.describe().T[["mean", "std"]].add_suffix("_te")
desc = pd.concat([desc_tr, desc_te], axis=1)
desc.round(3)


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

KL divergence aproximada por bins.


In [ ]:
def kl_hist(a, b, bins=20):
    a_h, _ = np.histogram(a, bins=bins, density=True)
    b_h, _ = np.histogram(b, bins=bins, density=True)
    a_h = a_h + 1e-9
    b_h = b_h + 1e-9
    return float(np.sum(a_h * np.log(a_h / b_h)))

cols = [c for c in X.columns if c != "y"]
kl = pd.Series({c: kl_hist(tr[c], te[c]) for c in cols}).sort_values()
kl


## 13. Visualizaciones explicativas

Bar chart KL.


In [ ]:
kl.plot.barh(color="#9C27B0", figsize=(7, 3))
plt.title("KL train vs test (bajo = misma distribución)")
plt.tight_layout()


## 14. Validaciones

KL < 1 para todas las features.


In [ ]:
assert kl.max() < 2.0, f"Drift fuerte: {kl}"
print("Drift OK")


## 15. Errores comunes

1. KL con bins muy pequeños.
2. No comparar la columna target.
3. Métricas en escala absoluta sin estandarizar.


## 16. Ejercicios propuestos

1. Implementa Wasserstein distance.
2. Visualiza un drift artificial añadiendo +5 °C al test.
3. Construye un detector que envíe alerta si KL > umbral.


## 17. Cómo se reutiliza con datos reales

Mismo notebook sobre dataset producción cada noche.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `07_case_G_data_quality_agents/04_agentes_especialistas_calidad.ipynb`.
- Documento web del caso: `docs/validation/data-quality.md`.
